# Build a SageMaker Pipeline with a ConditionStep that only registers a new model when validation AUC clears a threshold

A team's nightly retraining job re-trains the production model whether or not the validation AUC actually improved, and the SageMaker bill has tripled in six months. The fix is a SageMaker Pipeline with a ConditionStep that gates RegisterModel on `evaluation.auc >= 0.85`. When the new model is better, register. When it is worse, short-circuit the pipeline and emit a CloudWatch metric so the data team can see the no-op retrain.

You have one session to wire the five-step pipeline (Process, Train, Evaluate, Condition, RegisterModel), run it twice (good data and degraded data), and prove the ConditionStep does its job.

Five artifacts ship in this lab:

- A SageMaker Pipeline definition with five steps: ProcessingStep (split), TrainingStep (XGBoost), ProcessingStep (evaluate), ConditionStep, RegisterModelStep.
- A good run against a clean dataset that lands AUC above 0.85 and registers a Model Package.
- A degraded run against a noisy dataset that lands AUC below 0.85 and skips registration.
- A CloudWatch custom metric (`labrun/mla/ConditionalRetrainSkipped`) emitted by the evaluation script on the degraded run.
- A Model Package Group with exactly one PendingManualApproval Model Package after both runs complete.

**Time.** Plan for about 65 minutes hands-on. Pipeline definition and upsert are fast (under a minute). Each pipeline run is roughly 10 to 15 minutes wall-clock because ProcessingStep and TrainingStep each provision ml.m5.large instances. Budget 120 minutes if the first run fights back on property-file wiring.

**Cost.** This lab costs about four to ten cents per session. Pipelines orchestration is free; you pay only for the underlying step instances. Two runs of the pipeline (the degraded run skips the RegisterModelStep so it has four billable steps instead of five) on ml.m5.large at $0.115 per hour for about three minutes per step come out to roughly five cents combined. CloudWatch custom metrics are free below 10 per month; this lab uses one. The lesson the team learned at the end of this story: a $0.85 AUC gate saves them thousands of dollars a month at production retraining cadence.

This lab maps to AWS MLA-C01 Domain 3 (Deployment and Orchestration of ML Workflows, 22%) on SageMaker Pipelines, conditional retraining gates, and EventBridge-driven retraining (the same condition signal feeds an EventBridge rule in Lab 12).


In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus the SageMaker SDK for Pipelines and image URI lookups.

!pip install --quiet labrun-checks==0.6.0 sagemaker==2.232.0

In [ ]:
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern.

import atexit
import csv
import getpass
import io
import json
import random
import sys
import time
from datetime import datetime, timedelta, timezone

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "sagemaker-pipelines-conditional-retraining"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

PIPELINE_ROLE_NAME = f"labrun-{LAB_ID}-role"
PIPELINE_INLINE_POLICY = "labrun-mla-lab09-role-policy"
PIPELINE_NAME = f"labrun-{LAB_ID}"
PACKAGE_GROUP_NAME = f"labrun-{LAB_ID}-group"
CW_NAMESPACE = "labrun/mla"
CW_METRIC_NAME = "ConditionalRetrainSkipped"
AUC_THRESHOLD = 0.85

BUCKET_NAME = None
ACCOUNT_ID = None
PIPELINE_ROLE_ARN = None
XGBOOST_IMAGE_URI = None
SKLEARN_IMAGE_URI = None
GOOD_EXEC_ARN = None
DEGRADED_EXEC_ARN = None
SESSION_START = datetime.now(timezone.utc)

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# resolve account-derived names.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"MLA Batch 3 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
PIPELINE_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{PIPELINE_ROLE_NAME}"
print(f"Bucket: {BUCKET_NAME}")
print(f"Pipeline role ARN: {PIPELINE_ROLE_ARN}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, orphan scan. labrun-checks 0.6.0 has no
# adapter for SageMaker Pipelines, Pipeline Executions, Model Packages, or
# Model Package Groups; the cleanup cell tears those down manually BEFORE
# run_cleanup walks the manifest. The manifest below contains only
# adapter-supported types.
#
# Critical resources: in-flight Pipeline Executions. Each step in an
# in-flight execution bills per-second on its underlying instance. The
# cleanup cell calls stop_pipeline_execution on both executions before
# delete_pipeline.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="iam_role",
        id=PIPELINE_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {PIPELINE_ROLE_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list:
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first, or remove")
    print("them manually with the AWS CLI commands the cleanup cell prints.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Define and upsert the SageMaker Pipeline with five steps

Build the pipeline DAG as a Python SDK definition. The DAG has five steps in this order:

1. **ProcessingStep (`SplitStep`).** Reads `s3://{bucket}/raw/{good|degraded}.csv` and writes `train.csv` and `validation.csv` to a step output prefix.
2. **TrainingStep (`TrainStep`).** Runs the XGBoost built-in algorithm on the train and validation channels from the SplitStep, writes the model to a step output prefix.
3. **ProcessingStep (`EvalStep`).** Runs the evaluation script against the validation channel, scores the model, writes `evaluation.json` to a property-file path. The script also emits the `ConditionalRetrainSkipped` CloudWatch metric when AUC is below threshold.
4. **ConditionStep.** Gates on `evaluation.validation_metrics.auc.value >= 0.85`. The `IfSteps` list contains the RegisterModelStep. The `ElseSteps` list is empty.
5. **RegisterModelStep.** Registers the trained model in the Model Package Group with `ModelApprovalStatus=PendingManualApproval`.

Pre-baked helpers in this cell:

- The IAM role with SageMaker, S3, and CloudWatch permissions.
- The evaluation script, uploaded to S3 with the AUC computation and the CloudWatch put-metric for the no-op-retrain case.
- The good and degraded raw datasets, generated and uploaded.
- The pipeline definition object with all five steps wired up.

Your job: call `pipeline.upsert(role_arn=PIPELINE_ROLE_ARN, tags=[...])` to ship the pipeline.

In [ ]:
# NBVAL_SKIP
# Task 1: build the pipeline definition.

import sagemaker
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.parameters import ParameterString
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.model_step import ModelStep
from sagemaker.processing import (
    ProcessingInput,
    ProcessingOutput,
    ScriptProcessor,
)
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.model import Model
from sagemaker.model_metrics import MetricsSource, ModelMetrics

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
sm = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Bucket and tag.
try:
    s3.create_bucket(Bucket=BUCKET_NAME)
    print(f"Created bucket: {BUCKET_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
        print(f"Bucket {BUCKET_NAME} already exists, reusing.")
    else:
        raise
s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)

# Generate good.csv: 1000 rows, 8 features, label correlated with feature 0.
random.seed(42)
def gen_rows(n_rows: int, noise: float) -> str:
    buf = io.StringIO()
    writer = csv.writer(buf)
    writer.writerow(["label"] + [f"f{i}" for i in range(8)])
    for _ in range(n_rows):
        f0 = random.gauss(0, 1)
        feats = [f0] + [random.gauss(0, 1) for _ in range(7)]
        signal = f0 + random.gauss(0, noise)
        label = 1 if signal > 0 else 0
        writer.writerow([label] + [f"{v:.4f}" for v in feats])
    return buf.getvalue()

good_csv = gen_rows(1000, noise=0.3)
degraded_csv = gen_rows(1000, noise=4.0)
s3.put_object(Bucket=BUCKET_NAME, Key="raw/good.csv", Body=good_csv.encode("utf-8"))
s3.put_object(Bucket=BUCKET_NAME, Key="raw/degraded.csv", Body=degraded_csv.encode("utf-8"))
print("Uploaded raw/good.csv and raw/degraded.csv (1000 rows each).")

# IAM role with SageMaker service trust.
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "sagemaker.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}
try:
    iam.create_role(
        RoleName=PIPELINE_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="labrun mla lab 09 pipeline role",
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    print(f"Created role: {PIPELINE_ROLE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        print(f"Role {PIPELINE_ROLE_NAME} already exists, reusing.")
    else:
        raise

inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:PutObject", "s3:DeleteObject"],
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/*",
        },
        {
            "Effect": "Allow",
            "Action": "s3:ListBucket",
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}",
        },
        {
            "Effect": "Allow",
            "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
            "Resource": "*",
        },
        {
            "Effect": "Allow",
            "Action": "cloudwatch:PutMetricData",
            "Resource": "*",
        },
    ],
}
iam.put_role_policy(
    RoleName=PIPELINE_ROLE_NAME,
    PolicyName=PIPELINE_INLINE_POLICY,
    PolicyDocument=json.dumps(inline_policy),
)
print(f"Attached inline policy {PIPELINE_INLINE_POLICY}.")

print("Your IAM role is stuck in traffic, give it 10 seconds...")
time.sleep(10)

# Upload split script, evaluation script.
split_script = '''import argparse
import csv
import os
import random

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--input-data", required=True)
    args = p.parse_args()
    raw_dir = "/opt/ml/processing/input/raw"
    files = [f for f in os.listdir(raw_dir) if f.endswith(".csv")]
    rows = []
    header = None
    for fname in files:
        with open(os.path.join(raw_dir, fname)) as fh:
            reader = csv.reader(fh)
            header = next(reader)
            for r in reader:
                rows.append(r)
    random.seed(7)
    random.shuffle(rows)
    cut = int(0.7 * len(rows))
    train, val = rows[:cut], rows[cut:]
    os.makedirs("/opt/ml/processing/train", exist_ok=True)
    os.makedirs("/opt/ml/processing/validation", exist_ok=True)
    with open("/opt/ml/processing/train/train.csv", "w") as fh:
        w = csv.writer(fh)
        for r in train:
            w.writerow(r)
    with open("/opt/ml/processing/validation/validation.csv", "w") as fh:
        w = csv.writer(fh)
        for r in val:
            w.writerow(r)
    print(f"Wrote {len(train)} train rows, {len(val)} validation rows.")
'''
s3.put_object(Bucket=BUCKET_NAME, Key="code/split.py", Body=split_script.encode("utf-8"))

# Evaluation script reads val + model, computes AUC, writes evaluation.json.
# Emits CloudWatch metric on no-op-retrain case (auc below threshold).
eval_script = f'''import json
import os
import tarfile
import boto3
import xgboost as xgb
import numpy as np
from sklearn.metrics import roc_auc_score

AUC_THRESHOLD = {AUC_THRESHOLD}
CW_NAMESPACE = "{CW_NAMESPACE}"
CW_METRIC_NAME = "{CW_METRIC_NAME}"

model_dir = "/opt/ml/processing/model"
for f in os.listdir(model_dir):
    if f.endswith(".tar.gz"):
        with tarfile.open(os.path.join(model_dir, f)) as tar:
            tar.extractall(model_dir)

booster = xgb.Booster()
for f in os.listdir(model_dir):
    if f.startswith("xgboost-model"):
        booster.load_model(os.path.join(model_dir, f))
        break

val_dir = "/opt/ml/processing/validation"
rows = []
for fname in os.listdir(val_dir):
    if fname.endswith(".csv"):
        with open(os.path.join(val_dir, fname)) as fh:
            for line in fh:
                parts = line.strip().split(",")
                rows.append([float(p) for p in parts])
rows = np.array(rows)
y = rows[:, 0]
X = rows[:, 1:]
preds = booster.predict(xgb.DMatrix(X))
auc = float(roc_auc_score(y, preds))
print(f"Validation AUC: {{auc}}")

report = {{
    "validation_metrics": {{"auc": {{"value": auc, "standard_deviation": 0.0}}}},
}}
out = "/opt/ml/processing/evaluation/evaluation.json"
os.makedirs(os.path.dirname(out), exist_ok=True)
with open(out, "w") as fh:
    json.dump(report, fh)

if auc < AUC_THRESHOLD:
    cw = boto3.client("cloudwatch", region_name="{REGION}")
    cw.put_metric_data(
        Namespace=CW_NAMESPACE,
        MetricData=[{{"MetricName": CW_METRIC_NAME, "Value": 1, "Unit": "Count"}}],
    )
    print(f"AUC {{auc}} below threshold {{AUC_THRESHOLD}}; emitted CloudWatch metric.")
else:
    print(f"AUC {{auc}} above threshold {{AUC_THRESHOLD}}; no metric emitted.")
'''
s3.put_object(Bucket=BUCKET_NAME, Key="code/evaluate.py", Body=eval_script.encode("utf-8"))
print("Uploaded code/split.py and code/evaluate.py.")

# Image URIs.
XGBOOST_IMAGE_URI = sagemaker.image_uris.retrieve(framework="xgboost", region=REGION, version="1.5-1")
SKLEARN_IMAGE_URI = sagemaker.image_uris.retrieve(framework="sklearn", region=REGION, version="1.0-1")
print(f"XGBoost image: {XGBOOST_IMAGE_URI}")
print(f"Sklearn image: {SKLEARN_IMAGE_URI}")

# Try to create the Model Package Group up-front (idempotent).
try:
    sm.create_model_package_group(
        ModelPackageGroupName=PACKAGE_GROUP_NAME,
        ModelPackageGroupDescription="labrun mla lab 09 conditional registration",
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    print(f"Created model package group: {PACKAGE_GROUP_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "ValidationException" and "already exists" in str(e):
        print(f"Model package group {PACKAGE_GROUP_NAME} already exists, reusing.")
    else:
        raise

pipeline_session = PipelineSession(boto_session=boto3.Session(
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
))

input_data = ParameterString(
    name="InputDataUrl", default_value=f"s3://{BUCKET_NAME}/raw/good.csv"
)

split_processor = SKLearnProcessor(
    framework_version="1.0-1",
    instance_type="ml.m5.large",
    instance_count=1,
    role=PIPELINE_ROLE_ARN,
    sagemaker_session=pipeline_session,
)
split_step_args = split_processor.run(
    code=f"s3://{BUCKET_NAME}/code/split.py",
    inputs=[ProcessingInput(source=input_data, destination="/opt/ml/processing/input/raw")],
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/validation"),
    ],
    arguments=["--input-data", "raw"],
)
split_step = ProcessingStep(name="SplitStep", step_args=split_step_args)

xgb_estimator = Estimator(
    image_uri=XGBOOST_IMAGE_URI,
    instance_type="ml.m5.large",
    instance_count=1,
    role=PIPELINE_ROLE_ARN,
    output_path=f"s3://{BUCKET_NAME}/output/train/",
    sagemaker_session=pipeline_session,
    hyperparameters={
        "objective": "binary:logistic",
        "num_round": "30",
        "max_depth": "4",
        "eta": "0.2",
        "eval_metric": "auc",
    },
)
train_step_args = xgb_estimator.fit(
    inputs={
        "train": TrainingInput(
            s3_data=split_step.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv",
        ),
        "validation": TrainingInput(
            s3_data=split_step.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="text/csv",
        ),
    },
)
train_step = TrainingStep(name="TrainStep", step_args=train_step_args)

eval_processor = ScriptProcessor(
    image_uri=XGBOOST_IMAGE_URI,
    command=["python3"],
    instance_type="ml.m5.large",
    instance_count=1,
    role=PIPELINE_ROLE_ARN,
    sagemaker_session=pipeline_session,
)
eval_property = PropertyFile(
    name="EvaluationReport", output_name="evaluation", path="evaluation.json",
)
eval_step_args = eval_processor.run(
    code=f"s3://{BUCKET_NAME}/code/evaluate.py",
    inputs=[
        ProcessingInput(
            source=train_step.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=split_step.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            destination="/opt/ml/processing/validation",
        ),
    ],
    outputs=[ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation")],
)
eval_step = ProcessingStep(
    name="EvalStep", step_args=eval_step_args, property_files=[eval_property],
)

register_args = RegisterModel(
    name="RegisterModelStep",
    estimator=xgb_estimator,
    model_data=train_step.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.t2.medium", "ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name=PACKAGE_GROUP_NAME,
    approval_status="PendingManualApproval",
)

condition = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=eval_step.name,
        property_file=eval_property,
        json_path="validation_metrics.auc.value",
    ),
    right=AUC_THRESHOLD,
)
condition_step = ConditionStep(
    name="AucGateStep",
    conditions=[condition],
    if_steps=[register_args],
    else_steps=[],
)

pipeline = Pipeline(
    name=PIPELINE_NAME,
    parameters=[input_data],
    steps=[split_step, train_step, eval_step, condition_step],
    sagemaker_session=pipeline_session,
)

# YOUR CODE: call pipeline.upsert(role_arn=PIPELINE_ROLE_ARN,
# tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]) to ship the
# pipeline. The call is idempotent; calling it again with the same
# definition is safe.
print(f"Upserting pipeline {PIPELINE_NAME}...")

print(f"Pipeline {PIPELINE_NAME} is upserted.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: pipeline exists with five expected steps in correct order.

def checkpoint_1(session):
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            desc = sm_client.describe_pipeline(PipelineName=PIPELINE_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Pipeline {PIPELINE_NAME} was not found. Call "
                        f"pipeline.upsert(role_arn=PIPELINE_ROLE_ARN, tags=[...])."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        definition_str = desc.get("PipelineDefinition")
        if not definition_str:
            return CheckpointResult(
                status="fail",
                error_reason="Pipeline returned no PipelineDefinition body.",
            )
        defn = json.loads(definition_str)
        steps = defn.get("Steps", [])
        names = [s.get("Name") for s in steps]
        expected_top = ["SplitStep", "TrainStep", "EvalStep", "AucGateStep"]
        if names != expected_top:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Pipeline top-level steps are {names!r}, expected {expected_top!r}. "
                    f"The RegisterModelStep must be nested inside the ConditionStep IfSteps."
                ),
            )

        condition_step = steps[-1]
        if condition_step.get("Type") != "Condition":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Last step Type is {condition_step.get('Type')!r}, expected "
                    f"'Condition'."
                ),
            )
        if_steps = condition_step.get("Arguments", {}).get("IfSteps", [])
        if not any("Register" in (s.get("Name") or "") for s in if_steps):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "ConditionStep IfSteps does not contain a RegisterModel step. "
                    "The else-branch should stay empty; the if-branch carries RegisterModel."
                ),
            )

        tag_resp = sm_client.list_tags(ResourceArn=desc["PipelineArn"])
        tag_map = {t["Key"]: t["Value"] for t in tag_resp.get("Tags", [])}
        if tag_map.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Pipeline is missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. Pass tags=[...] "
                    f"to pipeline.upsert()."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The pipeline definition is built; only the upsert call is missing. Look at the SageMaker Python SDK reference for `Pipeline.upsert()` and pass `role_arn` plus `tags`.

</details>

<details><summary>Hint 2 (direction)</summary>

`pipeline.upsert(role_arn=..., tags=[{"Key": ..., "Value": ...}])` creates the pipeline if it does not exist and updates the definition if it does. The `role_arn` argument is the SageMaker execution role that owns the pipeline (PIPELINE_ROLE_ARN). The `tags` list applies labrun:lab-slug so the orphan scan picks it up.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1 (the call to upsert the pipeline):

```python
pipeline.upsert(
    role_arn=PIPELINE_ROLE_ARN,
    tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

</details>

**Wallet check.** Pipelines orchestration is free. So far you have paid for the IAM role (free), the S3 puts for the raw data and the two scripts (fractions of a penny), and the Model Package Group (free). Nothing has run on an instance yet. Damage so far: about $0.00. Your coffee still wins by infinity.

## Task 2: Start the pipeline against the good dataset and wait for success

Start an execution of the pipeline with `InputDataUrl=s3://{bucket}/raw/good.csv`. The execution takes about 10 to 15 minutes wall-clock: each ProcessingStep and TrainingStep provisions an ml.m5.large instance for roughly 2 to 3 minutes. The good dataset has a clean signal (feature 0 strongly correlates with the label) so the trained model lands AUC around 0.88 to 0.92, well above the 0.85 threshold.

After the run completes, the Model Package Group contains exactly one Model Package in PendingManualApproval status, registered by the RegisterModelStep that the ConditionStep's IfBranch executed.

In [ ]:
# NBVAL_SKIP
# Task 2: start the good-data pipeline run and wait for Succeeded.

# YOUR CODE: call pipeline.start(parameters={"InputDataUrl":
# f"s3://{BUCKET_NAME}/raw/good.csv"}) and assign the return value to
# `good_execution`. The return is a PipelineExecution object whose
# .arn attribute carries the execution ARN.
good_execution = None

GOOD_EXEC_ARN = good_execution.arn
print(f"Good-run execution started. ARN: {GOOD_EXEC_ARN}")

print("Pipeline is busy doing pipeline things, give it about 12 minutes...")
elapsed = 0
status = None
while elapsed < 1800:
    desc = sm.describe_pipeline_execution(PipelineExecutionArn=GOOD_EXEC_ARN)
    status = desc["PipelineExecutionStatus"]
    if status in ("Succeeded", "Failed", "Stopped"):
        break
    time.sleep(30)
    elapsed += 30
    if elapsed % 120 == 0:
        print(f"  {elapsed}s elapsed, status: {status}")

if status != "Succeeded":
    print(f"Good run ended with status {status}. Inspect SageMaker Studio for the failed step.")
    raise SystemExit(1)
print(f"Good run reached Succeeded.")

steps_resp = sm.list_pipeline_execution_steps(PipelineExecutionArn=GOOD_EXEC_ARN)
for s in steps_resp.get("PipelineExecutionSteps", []):
    print(f"  {s['StepName']}: {s['StepStatus']}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: good run Succeeded with all 5 step executions and a
# registered Model Package in the lab's Model Package Group.

def checkpoint_2(session):
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        if not GOOD_EXEC_ARN:
            return CheckpointResult(
                status="fail",
                error_reason="GOOD_EXEC_ARN is None. Start the good-data pipeline run.",
            )
        desc = sm_client.describe_pipeline_execution(PipelineExecutionArn=GOOD_EXEC_ARN)
        if desc["PipelineExecutionStatus"] != "Succeeded":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Good execution status is {desc['PipelineExecutionStatus']!r}, "
                    f"not Succeeded."
                ),
            )
        steps = sm_client.list_pipeline_execution_steps(PipelineExecutionArn=GOOD_EXEC_ARN)
        step_list = steps.get("PipelineExecutionSteps", [])
        if len(step_list) < 5:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Good execution produced {len(step_list)} step executions, expected 5. "
                    f"The RegisterModelStep should have run because AUC was above threshold."
                ),
            )
        non_success = [s for s in step_list if s["StepStatus"] != "Succeeded"]
        if non_success:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Some steps did not succeed: "
                    f"{[(s['StepName'], s['StepStatus']) for s in non_success]!r}"
                ),
            )

        pkgs = sm_client.list_model_packages(
            ModelPackageGroupName=PACKAGE_GROUP_NAME, MaxResults=10
        ).get("ModelPackageSummaryList", [])
        if not pkgs:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No Model Package found in group {PACKAGE_GROUP_NAME}. The good run "
                    f"should have registered one."
                ),
            )
        if pkgs[0].get("ModelApprovalStatus") != "PendingManualApproval":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Latest Model Package has ModelApprovalStatus="
                    f"{pkgs[0].get('ModelApprovalStatus')!r}, expected "
                    f"'PendingManualApproval'."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The pipeline takes a single parameter, `InputDataUrl`. Pass the parameters dict to `pipeline.start()` and capture the returned execution object so you can describe it later.

</details>

<details><summary>Hint 2 (direction)</summary>

`pipeline.start(parameters={"InputDataUrl": ...})` returns a `PipelineExecution` whose `.arn` is the ARN you then poll with `sm.describe_pipeline_execution`. The good URL is `s3://{BUCKET_NAME}/raw/good.csv`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2 (the pipeline start call):

```python
good_execution = pipeline.start(
    parameters={"InputDataUrl": f"s3://{BUCKET_NAME}/raw/good.csv"}
)
```

</details>

**Wallet check.** The good run executed five steps on ml.m5.large at $0.115/hour. ProcessingStep + TrainingStep + ProcessingStep + RegisterModelStep is roughly 4 billable steps at about 3 minutes each, which lands around $0.023. Damage so far: about $0.025. Your coffee remains comfortably ahead.

## Task 3: Confirm the EvaluationStep wrote an evaluation.json with AUC above threshold

The EvaluationStep declared a `PropertyFile` named `EvaluationReport` that captures `evaluation.json` from `/opt/ml/processing/evaluation/`. The ConditionStep reads `validation_metrics.auc.value` from that file via JsonGet.

Read the file directly from S3 and parse the AUC. The good run should land somewhere in the 0.85 to 0.95 range. The checkpoint asserts AUC is at or above the 0.85 threshold.

In [ ]:
# NBVAL_SKIP
# Task 3: read evaluation.json from the good run's EvalStep output.

good_eval_uri = None
steps = sm.list_pipeline_execution_steps(PipelineExecutionArn=GOOD_EXEC_ARN)
for s in steps.get("PipelineExecutionSteps", []):
    if s["StepName"] == "EvalStep":
        meta = s.get("Metadata", {}).get("ProcessingJob", {})
        job_arn = meta.get("Arn")
        if not job_arn:
            continue
        job_name = job_arn.split("/")[-1]
        job_desc = sm.describe_processing_job(ProcessingJobName=job_name)
        for out in job_desc.get("ProcessingOutputConfig", {}).get("Outputs", []):
            if out.get("OutputName") == "evaluation":
                good_eval_uri = out["S3Output"]["S3Uri"]
                break

if not good_eval_uri:
    print("Could not find the evaluation output prefix on the good run.")
    raise SystemExit(1)
print(f"Good run evaluation output prefix: {good_eval_uri}")

# YOUR CODE: parse good_eval_uri to bucket and key, then call
# s3.get_object(Bucket=..., Key=...+"/evaluation.json") and decode the
# body to JSON. Assign the parsed dict to `good_eval_report`.
good_eval_report = None

good_auc = good_eval_report["validation_metrics"]["auc"]["value"]
print(f"Good run validation AUC: {good_auc:.4f}")
print(f"Threshold: {AUC_THRESHOLD}")
print(f"Above threshold: {good_auc >= AUC_THRESHOLD}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: evaluation.json shows AUC at or above the 0.85 threshold.

def checkpoint_3(session):
    try:
        if good_eval_report is None:
            return CheckpointResult(
                status="fail",
                error_reason="good_eval_report is None. Read evaluation.json from S3.",
            )
        auc = good_eval_report.get("validation_metrics", {}).get("auc", {}).get("value")
        if auc is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "evaluation.json is missing validation_metrics.auc.value. "
                    "The ConditionStep JsonGet json_path expects that exact path."
                ),
            )
        if float(auc) < AUC_THRESHOLD:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Good run AUC {auc} is below threshold {AUC_THRESHOLD}. The good "
                    f"dataset should produce AUC well above 0.85. Re-run the good "
                    f"pipeline execution."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

S3 URIs look like `s3://bucket/key/with/slashes`. Strip the leading `s3://`, split on the first slash to separate bucket from key, then add `/evaluation.json` to the key.

</details>

<details><summary>Hint 2 (direction)</summary>

Use `s3.get_object(Bucket=..., Key=...)`, then read the response body via `resp["Body"].read()`. Decode to UTF-8 and pass to `json.loads`. The result is a dict shaped `{"validation_metrics": {"auc": {"value": ...}}}` (the exact path the ConditionStep's JsonGet uses).

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3 (parse the S3 URI and load evaluation.json):

```python
bucket_part, _, key_part = good_eval_uri.replace("s3://", "", 1).partition("/")
key = f"{key_part.rstrip('/')}/evaluation.json"
resp = s3.get_object(Bucket=bucket_part, Key=key)
good_eval_report = json.loads(resp["Body"].read().decode("utf-8"))
```

</details>

**Wallet check.** Reading a tiny JSON object from S3 is fractions of a penny. Total damage so far: about $0.025. The pipeline ran the EvaluationStep on ml.m5.large for three minutes, which was the last step in the good-run bill before the RegisterModelStep ran on a free orchestration call.

## Task 4: Start the pipeline against the degraded dataset and confirm RegisterModel was skipped

Now run the same pipeline against the degraded dataset. The degraded data has high noise (feature 0 is overwhelmed by Gaussian noise), so the trained model lands AUC around 0.55 to 0.65, well below the 0.85 threshold.

After the run, the pipeline status is `Succeeded` (the ConditionStep evaluating false does not fail the pipeline; the IfSteps are simply skipped). The step list has four step executions: SplitStep, TrainStep, EvalStep, AucGateStep, and no RegisterModelStep. The ConditionStep's metadata records `Outcome=False`.

In [ ]:
# NBVAL_SKIP
# Task 4: start the degraded-data pipeline run.

# YOUR CODE: call pipeline.start(parameters={"InputDataUrl":
# f"s3://{BUCKET_NAME}/raw/degraded.csv"}) and assign the return
# value to `degraded_execution`.
degraded_execution = None

DEGRADED_EXEC_ARN = degraded_execution.arn
print(f"Degraded-run execution started. ARN: {DEGRADED_EXEC_ARN}")

print("Pipeline is busy doing pipeline things, give it about 10 minutes...")
elapsed = 0
status = None
while elapsed < 1800:
    desc = sm.describe_pipeline_execution(PipelineExecutionArn=DEGRADED_EXEC_ARN)
    status = desc["PipelineExecutionStatus"]
    if status in ("Succeeded", "Failed", "Stopped"):
        break
    time.sleep(30)
    elapsed += 30
    if elapsed % 120 == 0:
        print(f"  {elapsed}s elapsed, status: {status}")

if status != "Succeeded":
    print(f"Degraded run ended with status {status}.")
    raise SystemExit(1)
print(f"Degraded run reached Succeeded.")

deg_steps = sm.list_pipeline_execution_steps(PipelineExecutionArn=DEGRADED_EXEC_ARN)
for s in deg_steps.get("PipelineExecutionSteps", []):
    outcome = s.get("Metadata", {}).get("Condition", {}).get("Outcome")
    suffix = f" [Condition.Outcome={outcome}]" if outcome else ""
    print(f"  {s['StepName']}: {s['StepStatus']}{suffix}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: degraded run Succeeded, ConditionStep Outcome=False,
# RegisterModelStep is absent, package group still has exactly one model.

def checkpoint_4(session):
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        if not DEGRADED_EXEC_ARN:
            return CheckpointResult(
                status="fail",
                error_reason="DEGRADED_EXEC_ARN is None. Start the degraded-data run.",
            )
        desc = sm_client.describe_pipeline_execution(
            PipelineExecutionArn=DEGRADED_EXEC_ARN
        )
        if desc["PipelineExecutionStatus"] != "Succeeded":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Degraded execution status is {desc['PipelineExecutionStatus']!r}, "
                    f"expected 'Succeeded' (the pipeline as a whole still succeeds even "
                    f"when the condition is false)."
                ),
            )
        steps = sm_client.list_pipeline_execution_steps(
            PipelineExecutionArn=DEGRADED_EXEC_ARN
        ).get("PipelineExecutionSteps", [])
        names = [s["StepName"] for s in steps]
        if any("Register" in n for n in names):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Degraded run executed a RegisterModel step ({names!r}). The "
                    f"ConditionStep should have evaluated false and skipped the if-branch."
                ),
            )
        cond = [s for s in steps if s["StepName"] == "AucGateStep"]
        if not cond:
            return CheckpointResult(
                status="fail",
                error_reason="AucGateStep is missing from the degraded run's step list.",
            )
        outcome = cond[0].get("Metadata", {}).get("Condition", {}).get("Outcome")
        if outcome != "False":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"AucGateStep Outcome is {outcome!r}, expected 'False'. The "
                    f"degraded dataset should produce AUC below 0.85."
                ),
            )
        pkgs = sm_client.list_model_packages(
            ModelPackageGroupName=PACKAGE_GROUP_NAME, MaxResults=10
        ).get("ModelPackageSummaryList", [])
        if len(pkgs) != 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Model Package Group has {len(pkgs)} packages, expected exactly 1 "
                    f"(the one from the good run). The degraded run must not register."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Same call shape as Task 2, different `InputDataUrl`. Point at `raw/degraded.csv` and capture the execution.

</details>

<details><summary>Hint 2 (direction)</summary>

The pipeline parameter is named `InputDataUrl`. Pass the degraded path and SageMaker re-uses the upserted pipeline definition; only the parameter changes between executions.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4 (the degraded run start call):

```python
degraded_execution = pipeline.start(
    parameters={"InputDataUrl": f"s3://{BUCKET_NAME}/raw/degraded.csv"}
)
```

</details>

**Wallet check.** The degraded run executed four billable steps (SplitStep, TrainStep, EvalStep; no RegisterModelStep) on ml.m5.large for about 3 minutes each, which lands around $0.018. Total session damage now sits at roughly $0.04 to $0.05. Your coffee, still ahead.

## Task 5: Confirm the CloudWatch custom metric was emitted on the degraded run

The evaluation script puts the `labrun/mla/ConditionalRetrainSkipped` metric whenever AUC falls below threshold. The degraded run should have emitted exactly one datapoint with Value=1. CloudWatch metrics show up in `get_metric_statistics` within roughly 5 minutes of being put.

Use the CloudWatch boto3 client to query the metric over the lab session window (from `SESSION_START` to now) with `Period=300` and `Statistics=["Sum"]`. At least one datapoint with Sum at or above 1 is expected.

In [ ]:
# NBVAL_SKIP
# Task 5: confirm CloudWatch picked up the metric the evaluation script emitted.

cw = boto3.client(
    "cloudwatch",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

print("CloudWatch is slowly waking up; metric ingest can take up to 5 minutes...")
metric_resp = None
for attempt in range(10):
    end = datetime.now(timezone.utc)
    # YOUR CODE: call cw.get_metric_statistics(Namespace=CW_NAMESPACE,
    # MetricName=CW_METRIC_NAME, StartTime=SESSION_START, EndTime=end,
    # Period=300, Statistics=["Sum"]) and assign the result to metric_resp.
    metric_resp = None
    if metric_resp and metric_resp.get("Datapoints"):
        break
    time.sleep(30)

if not metric_resp or not metric_resp.get("Datapoints"):
    print("No datapoints found yet for ConditionalRetrainSkipped.")
else:
    for dp in metric_resp["Datapoints"]:
        print(f"  Timestamp: {dp['Timestamp']}, Sum: {dp['Sum']}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: CloudWatch ConditionalRetrainSkipped metric has Sum >= 1.

def checkpoint_5(session):
    try:
        cw_client = boto3.client(
            "cloudwatch",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        end = datetime.now(timezone.utc)
        resp = cw_client.get_metric_statistics(
            Namespace=CW_NAMESPACE,
            MetricName=CW_METRIC_NAME,
            StartTime=SESSION_START - timedelta(minutes=5),
            EndTime=end + timedelta(minutes=5),
            Period=300,
            Statistics=["Sum"],
        )
        total = sum(dp.get("Sum", 0) for dp in resp.get("Datapoints", []))
        if total < 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"CloudWatch metric {CW_NAMESPACE}/{CW_METRIC_NAME} shows total Sum "
                    f"{total}, expected at least 1. The evaluation script puts the "
                    f"metric when AUC is below threshold; check the EvalStep logs."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(5, checkpoint_5)

<details><summary>Hint 1 (nudge)</summary>

CloudWatch returns datapoints as a list. You need to call `get_metric_statistics` and look at `resp["Datapoints"]`. The lab's `SESSION_START` is the start of the lab; passing it as StartTime widens the window enough.

</details>

<details><summary>Hint 2 (direction)</summary>

Pass `Namespace=CW_NAMESPACE`, `MetricName=CW_METRIC_NAME`, `StartTime=SESSION_START`, `EndTime=end`, `Period=300`, and `Statistics=["Sum"]`. The result is a dict with a `Datapoints` list; each item has `Sum` and `Timestamp`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 5 (the CloudWatch query):

```python
metric_resp = cw.get_metric_statistics(
    Namespace=CW_NAMESPACE,
    MetricName=CW_METRIC_NAME,
    StartTime=SESSION_START,
    EndTime=end,
    Period=300,
    Statistics=["Sum"],
)
```

</details>

**Wallet check.** CloudWatch custom metrics are free below 10 per month and this lab uses one. `GetMetricStatistics` is free. Total session damage: about $0.04 to $0.05. Your coffee continues to be the more expensive thing on your desk.

In [ ]:
# NBVAL_SKIP
# Cleanup. Manual SageMaker teardown first (no adapters in labrun-checks 0.6.0
# for pipelines, executions, model packages, or groups), then run_cleanup
# walks the iam_role + s3_bucket manifest.

sm_cleanup = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

manual_warnings = []


def _safe(label, action, fallback_cmd):
    try:
        action()
        print(f"Deleted {label}.")
    except ClientError as e:
        code = e.response["Error"]["Code"]
        if code in ("ValidationException", "ResourceNotFound", "ResourceNotFoundException"):
            print(f"{label} already gone, skipping.")
        else:
            manual_warnings.append(
                f"FAILED TO DELETE: {label}. Error: {e}. Run manually: {fallback_cmd}"
            )


# Stop any in-flight pipeline executions (critical: steps continue to bill).
for arn_label, arn in (("good", GOOD_EXEC_ARN), ("degraded", DEGRADED_EXEC_ARN)):
    if not arn:
        continue
    try:
        d = sm_cleanup.describe_pipeline_execution(PipelineExecutionArn=arn)
        if d["PipelineExecutionStatus"] == "Executing":
            sm_cleanup.stop_pipeline_execution(PipelineExecutionArn=arn)
            print(f"Requested stop on {arn_label} execution.")
            # Poll until Stopped.
            elapsed = 0
            while elapsed < 180:
                d2 = sm_cleanup.describe_pipeline_execution(PipelineExecutionArn=arn)
                if d2["PipelineExecutionStatus"] in ("Stopped", "Succeeded", "Failed"):
                    break
                time.sleep(10)
                elapsed += 10
    except ClientError as e:
        if e.response["Error"]["Code"] not in ("ValidationException",):
            manual_warnings.append(f"FAILED to stop {arn_label} execution: {e}")

# Delete the pipeline itself.
_safe(
    f"pipeline {PIPELINE_NAME}",
    lambda: sm_cleanup.delete_pipeline(PipelineName=PIPELINE_NAME),
    f"aws sagemaker delete-pipeline --pipeline-name {PIPELINE_NAME}",
)

# Delete model packages and the group.
try:
    pkgs = sm_cleanup.list_model_packages(
        ModelPackageGroupName=PACKAGE_GROUP_NAME, MaxResults=50
    ).get("ModelPackageSummaryList", [])
except ClientError:
    pkgs = []
for p in pkgs:
    arn = p["ModelPackageArn"]
    _safe(
        f"model package {arn}",
        lambda a=arn: sm_cleanup.delete_model_package(ModelPackageName=a),
        f"aws sagemaker delete-model-package --model-package-name {arn}",
    )
_safe(
    f"model package group {PACKAGE_GROUP_NAME}",
    lambda: sm_cleanup.delete_model_package_group(
        ModelPackageGroupName=PACKAGE_GROUP_NAME
    ),
    f"aws sagemaker delete-model-package-group --model-package-group-name {PACKAGE_GROUP_NAME}",
)

# Hand off to run_cleanup for IAM role + S3 bucket.
result = run_cleanup(CLEANUP_MANIFEST)

for warning in manual_warnings:
    print(warning)
for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)

failed_ids = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 2  # 2 in-flight pipeline executions tracked critical
critical_destroyed = critical_total  # stop_pipeline_execution is idempotent on completed runs
standard_total = len(CLEANUP_MANIFEST)
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids) + len(manual_warnings)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed} of {critical_total}")
print(f"Standard resources destroyed: {standard_destroyed} of {standard_total}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

final_status = "clean" if (failed_count == 0 and result.status == "clean") else "dirty"
cleanup(status=final_status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.04 to $0.10.** Two pipeline runs on ml.m5.large at $0.115 per hour for three minutes per step come out to roughly five cents combined. The CloudWatch custom metric is free. The S3 storage and PUT operations are fractions of a penny. CodePipeline orchestration is free; you pay only for the underlying step instances. Check your AWS Billing console in 24 hours to confirm zero ongoing charges from this lab.

## Reflection

These are not graded. They are for you.

1. Walk through the value of the ConditionStep in a nightly retraining pipeline. The team's old setup retrains unconditionally and pays for compute every night even when the new model is worse than production. The new pattern gates on validation AUC and short-circuits when the gate fails. Where else in the AWS MLOps pattern do conditional gates show up (think CodePipeline approval actions, EventBridge rules with event-pattern matching, IAM `Condition` clauses), and what is the shared design principle behind them?

2. PropertyFiles wire data between SageMaker Pipeline steps. Walk through how the EvaluationStep writes evaluation.json to S3, the property file declaration captures the path, the ConditionStep reads `validation_metrics.auc.value` via JsonGet, and the condition compares the value to the threshold. What other property files would a production pipeline write (think feature drift scores, fairness metrics, model size in MB), and how would each one feed a separate ConditionStep in a multi-gate retraining workflow?

## Exam-Style Practice

**Question 1 (Domain 3, SageMaker Pipelines vs other orchestrators):**

A team is choosing between SageMaker Pipelines, AWS Step Functions, MWAA (Managed Workflows for Apache Airflow), and AWS Batch for an ML retraining workflow that has 5 steps: data validation, feature transformation, model training, model evaluation against a metric threshold, and conditional model registration. The team wants tight SageMaker integration, native support for Model Registry, and a single AWS-native orchestrator. Which fits?

A. SageMaker Pipelines. Built-in support for ProcessingStep, TrainingStep, RegisterModelStep, and ConditionStep with property-file wiring between steps; native Model Registry integration; free orchestration with per-step instance billing.

B. AWS Step Functions. The standard AWS orchestrator with Lambda task states for each step.

C. MWAA. Apache Airflow with native AWS hooks for SageMaker.

D. AWS Batch. Submit each step as a Batch job.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: SageMaker Pipelines is purpose-built for ML workflows. The five-step pattern in the scenario maps exactly to the SDK's built-in step types (ProcessingStep, TrainingStep, ConditionStep, RegisterModelStep). Property-file wiring between steps and native Model Registry integration are first-class features. Orchestration is free; you pay only for per-step instance compute.
- B is partially right: Step Functions can orchestrate SageMaker, but the team would build the property-file plumbing and Model Registry integration themselves. Higher operational overhead for the same outcome.
- C is wrong on "AWS-native": MWAA is Apache Airflow on AWS. It works but adds the operational overhead of an Airflow cluster ($0.49/hour for the smallest size, continuously billed) for a workflow that SageMaker Pipelines handles natively for free.
- D is wrong on fit: AWS Batch is a job-submission service for batch compute (think rendering, simulation). It does not have native ML-step types and does not integrate with Model Registry.

</details>

**Question 2 (Domain 3, ConditionStep semantics):**

A SageMaker Pipeline has an EvaluationStep that writes `evaluation.json` containing `{"validation_metrics": {"auc": {"value": 0.82}}}`. The ConditionStep is configured with `ConditionGreaterThanOrEqualTo(left=JsonGet(step_name="eval", property_file=eval_pf, json_path="validation_metrics.auc.value"), right=0.85)` and `if_steps=[RegisterModelStep], else_steps=[]`. What happens when the pipeline runs against this evaluation output?

A. The pipeline fails because the ConditionStep evaluates false and there is no else-branch action.

B. The RegisterModelStep executes because ConditionStep defaults to running if_steps when the path resolution fails.

C. The pipeline succeeds, the ConditionStep evaluates false, the RegisterModelStep is skipped, and no Model Package is registered.

D. The pipeline retries the EvaluationStep until the metric clears the threshold.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: a ConditionStep evaluating false with an empty else-branch is the AWS-supported pattern for "skip the if_steps." The pipeline succeeds; nothing fails.
- B is wrong: ConditionStep does not default to running if_steps on path-resolution failure. If the JsonGet path fails to resolve, the step itself fails with a JsonGet error.
- C is correct: 0.82 is less than 0.85, the condition evaluates false, the if_steps (RegisterModelStep) are skipped, the else_steps are empty so nothing else runs, and the pipeline execution succeeds with the ConditionStep's Outcome field set to false.
- D is wrong: SageMaker Pipelines does not retry steps based on condition outcomes. Retries are configured per step via `retry_policies` and are for transient failures, not for re-running a step until a metric improves.

</details>